# Chapter 6 &mdash; Worked Example: Union, Minimization, Two Predicates

**Concept 11 of the Chapter 6 decomposition:** *Worked Example: Union, Minimization, and the Two Comparison Predicates*

Build two DFA, union them, minimize, and watch `iso_dfa` say False while `langeq_dfa` says True.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6-DFAOps/Concept-Worked-Union-Minimization/Concept-Worked-Union-Minimization.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


The full pipeline on one example:

1. build two DFA;
2. **union** them &mdash; the product construction inflates the state count;
3. **prune** the unreachable pairs;
4. **minimize** &mdash; the count collapses;
5. compare with **both** predicates.

The punchline is step 5: the raw union and the minimized union are
**language-equivalent** but **not isomorphic**. Confusing the two predicates is the
most common mistake in this chapter.

## 2. Definitions

### Two machines to combine

In [ ]:
even0 = md2mc('''DFA
IF : 0 -> Od
IF : 1 -> IF
Od : 0 -> IF
Od : 1 -> Od
''')
end1 = md2mc('''DFA
I : 0 -> I
I : 1 -> F
F : 0 -> I
F : 1 -> F
''')

### The pipeline, as one function

In [ ]:
def pipeline(A, B):
    U  = union_dfa(A, B)
    Up = pruneUnreach(U)
    Um = min_dfa(Up)
    return U, Up, Um

<!-- nav-strip -->

---

&larr;&nbsp;[Ch6&nbsp;10.&nbsp;Minimization as a Fixed-Point Computation: `fixptDist`](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6-DFAOps/Concept-Fixed-Point-Minimization/Concept-Fixed-Point-Minimization.ipynb) &nbsp;&middot;&nbsp; [**Chapter 6** index](https://github.com/ganeshutah/Jove/blob/master/Chapter6-DFAOps/README.md) &nbsp;&middot;&nbsp; [Ch6&nbsp;12.&nbsp;DeMorgan's Law for DFA, Verified by Isomorphism](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6-DFAOps/Concept-DeMorgan-For-DFA/Concept-DeMorgan-For-DFA.ipynb)&nbsp;&rarr;

---

## 3. Tests

Sizes at each stage.

In [ ]:
U, Up, Um = pipeline(even0, end1)
print("union      : %2d states" % len(U["Q"]))
print("pruned     : %2d states" % len(Up["Q"]))
print("minimized  : %2d states" % len(Um["Q"]))
assert len(Um["Q"]) <= len(Up["Q"]) <= len(U["Q"])

All three accept the same language.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(11) for p in product('01', repeat=k)]
spec = lambda s: (s.count('0') % 2 == 0) or s.endswith('1')
for name, X in [('union', U), ('pruned', Up), ('minimized', Um)]:
    bad = [s for s in strs if accepts_dfa(X, s) != spec(s)]
    print("%-10s mismatches : %d" % (name, len(bad)))
    assert not bad

**The punchline:** equivalent but not isomorphic.

In [ ]:
print("langeq_dfa(U, Um) :", langeq_dfa(U, Um))
print("iso_dfa(U, Um)    :", iso_dfa(U, Um))
assert langeq_dfa(U, Um)
assert not iso_dfa(U, Um)
print("\nSame language (%d states vs %d) -- different machines."
      % (len(U["Q"]), len(Um["Q"])))

Minimizing both sides restores isomorphism, as Myhill&ndash;Nerode promises.

In [ ]:
print("iso_dfa(min(U), min(Um)) :", iso_dfa(min_dfa(U), min_dfa(Um)))
assert iso_dfa(min_dfa(U), min_dfa(Um))

## 4. Animation

The minimized union &mdash; the whole pipeline's output.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(min_dfa(pruneUnreach(union_dfa(even0, end1))), FuseEdges=True)

## 5. Exercises


1. Run the same pipeline with `intersect_dfa`. How do the sizes compare?
2. Does pruning before minimizing change the final answer? Does it change the cost?
3. Find two DFA whose union does **not** shrink at all under minimization.

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 255 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter6-DFAOps/Concept-Worked-Union-Minimization')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')